# Omnibus — geo insights

`features.parquet` now ships with `stop_lat` / `stop_lon` (joined in from GTFS — see [docs/GTFS.md](../docs/GTFS.md) for source + coverage). This notebook puts five questions onto the map.

Same caveats as `01_first_insights.ipynb` + one extra:
- Drop `Daten_Linie_1_2024-09_2025-08` (overlap with event windows).
- Filter `|delay_arr_s| < 7200` (midnight wrap, §9).
- **~4.5% of stop-events lack coordinates** (see docs/GTFS.md). The map filters them out — they don't disappear from the underlying delay analysis, they just can't be plotted.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

df = pl.read_parquet("../data/parquet/features.parquet")
ev = df.filter(pl.col("source_window") != "Daten_Linie_1_2024-09_2025-08")
clean = ev.filter(pl.col("delay_arr_s").abs() < 7200)

geo = clean.filter(pl.col("stop_lat").is_not_null())
print(f"clean rows: {clean.height:,}")
print(f"with geo:   {geo.height:,}  ({geo.height/clean.height*100:.1f}%)")
print(f"unique geocoded stops: {geo['stop_name'].n_unique()}")

## Q1 — Where does Regensburg's unreliability live? (city map)

Per-stop median + σ, plotted on map. Hotspot labeling for the top 10 σ stops.

In [ ]:
per_stop = (
    # Exclude each trip's terminus (is_terminus, from assemble.py): a terminus's σ is
    # layover/recovery, not en-route reliability, and would read as a false hotspot.
    # Q2 (distance bands) consumes this frame, so it inherits the same clean basis.
    geo.filter(pl.col("productive_arr") & ~pl.col("is_terminus"))
    .group_by("stop_name").agg(
        pl.col("delay_arr_s").std().alias("sigma"),
        pl.col("delay_arr_s").median().alias("med"),
        pl.col("stop_lat").first(),
        pl.col("stop_lon").first(),
        pl.len().alias("n"))
    .filter(pl.col("n") > 1000)
)
print(f"stops on map: {per_stop.height}")

fig, ax = plt.subplots(figsize=(11, 10))
sc = ax.scatter(per_stop["stop_lon"], per_stop["stop_lat"],
                c=per_stop["sigma"], s=per_stop["n"]/400,
                cmap="RdYlGn_r", alpha=0.78, edgecolor="black", linewidth=0.3,
                vmin=80, vmax=350)

# label top 10 by sigma
top = per_stop.sort("sigma", descending=True).head(10)
for r in top.iter_rows(named=True):
    ax.annotate(r["stop_name"], (r["stop_lon"], r["stop_lat"]),
                fontsize=7, xytext=(4,4), textcoords="offset points")

ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"RVV stops — color = σ(delay), size = event count  ({per_stop.height} stops)")
plt.colorbar(sc, label="σ arrival delay [s]")
ax.set_aspect(1/0.66)  # rough latitude correction at 49°N
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

**Read:** Hotspots cluster geographically — the **northern outer ring** (Wutzlhofen, Wetterstation, Harzstraße) is a single delay-prone neighbourhood, not random scatter, and a second pocket sits on the **western Graß corridor** (Graß, Graß Nord, Roter-Brach-Weg). Because each trip's terminus is now excluded, these are genuine *en-route* stops, not layover endpoints — the old chart-toppers (Pentling, Sallerner Berg) were trip ends and have correctly dropped out. The Donau river bend is visible as the empty horizontal strip — no stops on water.

The city centre (high dot density around 12.09–12.11) is mostly green: high traffic but predictable. That matches Q3 in the deeper-dive notebook (city is best during week, friday afternoon being exception).

## Q2 — Does unreliability grow with distance from the centre?

Center = Hauptbahnhof (49.0125, 12.0992). Compute haversine distance per stop, bucket, look at σ.

In [ ]:
HBF_LAT, HBF_LON = 49.0125, 12.0992

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

per_stop_dist = per_stop.with_columns(
    dist_km=pl.Series(haversine_km(HBF_LAT, HBF_LON,
                                    per_stop["stop_lat"].to_numpy(),
                                    per_stop["stop_lon"].to_numpy()))
).with_columns(
    dist_bin=pl.when(pl.col("dist_km") < 1).then(pl.lit("0  <1 km (Altstadt)"))
              .when(pl.col("dist_km") < 2).then(pl.lit("1  1–2 km"))
              .when(pl.col("dist_km") < 4).then(pl.lit("2  2–4 km"))
              .when(pl.col("dist_km") < 7).then(pl.lit("3  4–7 km"))
              .otherwise(pl.lit("4  ≥7 km (Landkreis)"))
)
by_dist = (per_stop_dist.group_by("dist_bin")
    .agg(pl.col("sigma").median().alias("sigma_med"),
         pl.col("sigma").quantile(0.9).alias("sigma_p90"),
         pl.col("med").median().alias("delay_med"),
         pl.len().alias("n_stops"))
    .sort("dist_bin"))
by_dist

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(by_dist))
ax.bar(x - 0.2, by_dist["sigma_med"], width=0.4, color="#c0392b", label="median σ of stops in band")
ax.bar(x + 0.2, by_dist["sigma_p90"], width=0.4, color="#8e44ad", label="p90 σ of stops in band")
ax.set_xticks(x); ax.set_xticklabels(by_dist["dist_bin"], rotation=15, ha="right")
ax.set_ylabel("σ(arrival delay) [s]")
ax.set_title("Unreliability grows with distance from Hauptbahnhof")
ax.legend(); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

**Read:** Stops within 1 km of HBF have median σ ≈ 119s. It climbs to ≈ 144s in the 4–7 km band (p90 stop ≈ 175s) before the sparse ≥7 km bin (only ~6 stops) falls back. So unreliability **grows with distance out to the edge of the built-up network** — a natural consequence of variance propagation (Q1 in deeper-dive): far stops are downstream of more chances to accumulate delay. With each trip's terminus excluded the gradient is gentler than the raw view (terminus layover used to inflate the periphery) but still clean.

This is a clean story for the pitch: "the further you live from the centre, the less predictable your bus is — *measurably*."

## Q3 — Map of the flood: where did the June 2024 damage actually land?

Per-stop delta = (flood-week median delay) − (Oct-2024 baseline median). Positive = stop got *worse* during the flood.

In [ ]:
flood = (geo.filter(pl.col("source_window").str.starts_with("26.05.2024"))
    .filter(pl.col("productive_arr"))
    .group_by("stop_name").agg(
        pl.col("delay_arr_s").median().alias("med_flood"),
        pl.col("stop_lat").first(), pl.col("stop_lon").first(),
        pl.len().alias("n_flood")))
base = (geo.filter(pl.col("source_window") == "06.10.2024_19.10.2024_ITCS")
    .filter(pl.col("productive_arr"))
    .group_by("stop_name").agg(
        pl.col("delay_arr_s").median().alias("med_base"),
        pl.len().alias("n_base")))

delta = flood.join(base, on="stop_name", how="inner").filter(
    (pl.col("n_flood") > 100) & (pl.col("n_base") > 100)
).with_columns(delta=pl.col("med_flood") - pl.col("med_base"))

print(f"stops with both windows: {delta.height}")
print(f"delta percentiles: p10={delta['delta'].quantile(0.1):.0f}s  median={delta['delta'].median():.0f}s  p90={delta['delta'].quantile(0.9):.0f}s")

fig, ax = plt.subplots(figsize=(11, 10))
sc = ax.scatter(delta["stop_lon"], delta["stop_lat"], c=delta["delta"], s=40,
                cmap="RdBu_r", alpha=0.85, edgecolor="black", linewidth=0.3,
                vmin=-60, vmax=60)
worst = delta.sort("delta", descending=True).head(8)
for r in worst.iter_rows(named=True):
    ax.annotate(f"{r['stop_name']} +{r['delta']:.0f}s", (r["stop_lon"], r["stop_lat"]),
                fontsize=7, xytext=(4,4), textcoords="offset points")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Flood-week delta in median delay vs Oct baseline — red = got worse, blue = got better")
plt.colorbar(sc, label="Δ median delay [s]")
ax.set_aspect(1/0.66); ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

**Read:** The flood damage **isn't** geographically uniform. A concentrated band of red appears along corridors that hug the Donau. Some inland stops actually got *faster* during the flood (blue) — likely because their feeder traffic was redirected or commuters stayed home. This is the kind of map Scene C should drive in real-time.

## Q4 — Klinikum is the hospital — what's special about its corridor?

Pick the worst-performing single hotspot from Q1 (`Klinikum`) and see which lines call there + how they compare to network baseline at that stop.

In [ ]:
k = (geo.filter(pl.col("productive_arr") & (pl.col("stop_name") == "Klinikum"))
    .group_by("line").agg(
        pl.col("delay_arr_s").std().alias("sigma"),
        pl.col("delay_arr_s").median().alias("med"),
        pl.col("delay_arr_s").quantile(0.9).alias("p90"),
        pl.len().alias("n"))
    .filter(pl.col("n") > 100)
    .sort("sigma", descending=True))
k

**Read:** Multiple lines converge at Klinikum and each one has its own variance profile — Line 8 is the chief offender. This is a per-line, per-stop intervention opportunity that the city could action (signal-pre-emption for the worst line at the worst stop).

## Q5 — Trace a single trip's delay along the city, on the map

Pick the worst trip on Line X4 in the Oct 2024 baseline. Plot its stops in geo order, color each by the delay it incurred *between* that stop and the previous one.

In [ ]:
x4 = (geo.filter((pl.col("line")=="X4")
                 & (pl.col("source_window")=="06.10.2024_19.10.2024_ITCS")
                 & pl.col("productive_arr"))
    .sort("trip_id","stop_seq"))

# pick the trip with the worst final-stop delay
last_per_trip = (x4.group_by("trip_id").agg(
    pl.col("delay_arr_s").last().alias("end_delay"),
    pl.col("stop_seq").max().alias("max_seq"))
    .filter(pl.col("max_seq") > 10))
worst_trip_id = last_per_trip.sort("end_delay", descending=True).row(0, named=True)["trip_id"]

trip = x4.filter(pl.col("trip_id") == worst_trip_id).sort("stop_seq")
print(f"trip {worst_trip_id}: {trip.height} stops, end delay {trip['delay_arr_s'].last()}s")

fig, ax = plt.subplots(figsize=(10, 9))
ax.plot(trip["stop_lon"], trip["stop_lat"], color="#888", linewidth=1.5, alpha=0.6, zorder=1)
sc = ax.scatter(trip["stop_lon"], trip["stop_lat"], c=trip["delay_arr_s"], s=70,
                cmap="RdYlGn_r", edgecolor="black", linewidth=0.4, zorder=2,
                vmin=trip["delay_arr_s"].min(), vmax=trip["delay_arr_s"].max())
for i, r in enumerate(trip.iter_rows(named=True)):
    if i % 3 == 0 or i == trip.height - 1:
        ax.annotate(f"{r['stop_name']} ({r['delay_arr_s']}s)",
                    (r["stop_lon"], r["stop_lat"]),
                    fontsize=7, xytext=(5,5), textcoords="offset points")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"Worst X4 trip in Oct 2024 baseline — color = delay at each stop")
plt.colorbar(sc, label="delay [s]")
ax.set_aspect(1/0.66); ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

**Read:** A single bad trip on X4, traced from origin to terminus. Watch the colour transition: stops start green (on time at the depot), then go red as the trip accumulates delay. The map shows *where* the bus actually slips — and you can read off the worst street segment by inspection.

This is the Scene B → Scene A bridge: variance-ranking finds the bad line, geo-tracing finds the bad **place** on that line.

## What's still missing for full geo

- **Route shapes** (`shapes.txt` from GTFS) — the actual street geometry between stops. The free gtfs.de feed omits these; would need paid feed or RVV direct. Without them we can only draw stop-to-stop straight lines.
- **OSRM / map-matching** — to attribute each between-stops segment to specific street IDs. Unblocks Scene A's "this intersection costs N hours/week" claims.
- **Recover the 4.5% missing coords** — see [docs/GTFS.md](../docs/GTFS.md) for the 7 real-stop names we couldn't match.